In [1]:
import pandas as pd
import numpy as np
import os

PROC = "bluestock/data/processed"
DASH = "bluestock/dashboard"
os.makedirs(DASH, exist_ok=True)

df_fund  = pd.read_csv(f"{PROC}/01_fund_master_clean.csv")
df_nav   = pd.read_csv(f"{PROC}/02_nav_history_clean.csv",  parse_dates=["date"])
df_aum   = pd.read_csv(f"{PROC}/03_aum_clean.csv",          parse_dates=["date"])
df_sip   = pd.read_csv(f"{PROC}/04_sip_clean.csv",          parse_dates=["month"])
df_cat   = pd.read_csv(f"{PROC}/05_category_clean.csv",     parse_dates=["month"])
df_fol   = pd.read_csv(f"{PROC}/06_folio_clean.csv",        parse_dates=["month"])
df_txn   = pd.read_csv(f"{PROC}/08_transactions_clean.csv", parse_dates=["transaction_date"])
df_bm    = pd.read_csv(f"{PROC}/10_benchmark_clean.csv",    parse_dates=["date"])
df_score = pd.read_csv(f"{PROC}/fund_scorecard.csv")
df_master= pd.read_csv(f"{PROC}/performance_master.csv")

print("All data loaded")

All data loaded


In [2]:
# Main fund performance table for Power BI
pbi_funds = df_master[[
    "amfi_code","scheme_name","fund_house","category",
    "sub_category","cagr_1y_pct","cagr_3y_pct","cagr_5y_pct",
    "sharpe_ratio","sortino_ratio","alpha_pct","beta",
    "max_drawdown_pct","var_95_pct","ann_std_pct","composite_score"
]].copy()

pbi_funds = pbi_funds.merge(
    df_fund[["amfi_code","expense_ratio_pct","risk_category",
             "fund_manager","benchmark","plan"]], on="amfi_code")

pbi_funds.to_csv(f"{DASH}/pbi_fund_performance.csv", index=False)
print(f" Fund Performance  : {len(pbi_funds)} rows")
print(pbi_funds.head(3).to_string(index=False))

 Fund Performance  : 40 rows
 amfi_code                                        scheme_name       fund_house category   sub_category  cagr_1y_pct  cagr_3y_pct  cagr_5y_pct  sharpe_ratio  sortino_ratio  alpha_pct    beta  max_drawdown_pct  var_95_pct  ann_std_pct  composite_score  expense_ratio_pct risk_category    fund_manager                    benchmark    plan
    100016          HDFC Top 100 Fund - Regular Plan - Growth HDFC Mutual Fund   Equity      Large Cap        -2.22         1.29         2.32        -0.202         -0.351     3.7476 -0.0583            -24.73     -1.4364       14.548            21.12               1.55      Moderate    Rahul Baijal                NIFTY 100 TRI Regular
    100025       HDFC Short Term Debt Fund - Regular - Growth HDFC Mutual Fund     Debt Short Duration         3.70         3.92         3.91        -0.567         -0.942     4.2818  0.0012             -4.31     -0.3793        3.905            23.62               0.56           Low    Anil Bamboli 

In [3]:
# Monthly average NAV per fund (lighter than daily for Power BI)
df_nav["year_month"] = df_nav["date"].dt.to_period("M").astype(str)

pbi_nav = df_nav.groupby(["amfi_code","year_month"]).agg(
    avg_nav   = ("nav",          "mean"),
    min_nav   = ("nav",          "min"),
    max_nav   = ("nav",          "max"),
    avg_return= ("daily_return_pct", "mean"),
).reset_index()

pbi_nav = pbi_nav.merge(
    df_fund[["amfi_code","scheme_name","fund_house","category"]],
    on="amfi_code")

pbi_nav.to_csv(f"{DASH}/pbi_monthly_nav.csv", index=False)
print(f"Monthly NAV       : {len(pbi_nav):,} rows")

Monthly NAV       : 2,120 rows


In [4]:
pbi_aum = df_aum.copy()
pbi_aum["year"]    = pbi_aum["date"].dt.year
pbi_aum["quarter"] = pbi_aum["date"].dt.quarter
pbi_aum["year_q"]  = pbi_aum["date"].dt.to_period("Q").astype(str)

pbi_aum.to_csv(f"{DASH}/pbi_aum_trends.csv", index=False)
print(f"AUM Trends        : {len(pbi_aum)} rows")

# KPI — Total industry AUM latest
latest_aum = pbi_aum.sort_values("date").groupby("fund_house").last()
total_aum  = latest_aum["aum_lakh_crore"].sum()
print(f"   Total Industry AUM : ₹{total_aum:.2f} Lakh Crore")

AUM Trends        : 90 rows
   Total Industry AUM : ₹62.74 Lakh Crore


In [5]:
pbi_sip = df_sip.copy()
pbi_sip["year"]  = pbi_sip["month"].dt.year
pbi_sip["month_name"] = pbi_sip["month"].dt.strftime("%b %Y")

pbi_sip.to_csv(f"{DASH}/pbi_sip_trends.csv", index=False)
print(f"SIP Trends        : {len(pbi_sip)} rows")
print(f"   Peak SIP          : ₹{pbi_sip['sip_inflow_crore'].max():,} Cr")
print(f"   Latest SIP Accounts: {pbi_sip['active_sip_accounts_crore'].iloc[-1]} Cr")

SIP Trends        : 48 rows
   Peak SIP          : ₹31,002 Cr
   Latest SIP Accounts: 9.35 Cr


In [6]:
pbi_txn = df_txn.copy()
pbi_txn["year"]       = pbi_txn["transaction_date"].dt.year
pbi_txn["month"]      = pbi_txn["transaction_date"].dt.to_period("M").astype(str)
pbi_txn["month_name"] = pbi_txn["transaction_date"].dt.strftime("%b %Y")

pbi_txn = pbi_txn.merge(
    df_fund[["amfi_code","scheme_name","fund_house","category"]],
    on="amfi_code", how="left")

pbi_txn.to_csv(f"{DASH}/pbi_transactions.csv", index=False)
print(f" Transactions      : {len(pbi_txn):,} rows")

# Quick stats
print(f"\n   By Type:")
print(pbi_txn["transaction_type"].value_counts().to_string())
print(f"\n   By City Tier:")
print(pbi_txn["city_tier"].value_counts().to_string())

 Transactions      : 32,778 rows

   By Type:
transaction_type
Sip           19716
Lumpsum        8095
Redemption     4967

   By City Tier:
city_tier
T30    21719
B30    11059


In [7]:
pbi_cat = df_cat.copy()
pbi_cat["year"]       = pbi_cat["month"].dt.year
pbi_cat["month_name"] = pbi_cat["month"].dt.strftime("%b %Y")

pbi_cat.to_csv(f"{DASH}/pbi_category_inflows.csv", index=False)
print(f" Category Inflows  : {len(pbi_cat)} rows")

 Category Inflows  : 144 rows


In [8]:
pbi_bm = df_bm.copy()
pbi_bm["year"]       = pbi_bm["date"].dt.year
pbi_bm["month"]      = pbi_bm["date"].dt.to_period("M").astype(str)
pbi_bm["year_month"] = pbi_bm["date"].dt.to_period("M").astype(str)

# Normalized to 100
pbi_bm_norm = []
for name, grp in pbi_bm.groupby("index_name"):
    grp = grp.sort_values("date").copy()
    grp["normalized"] = (grp["close_value"] / grp["close_value"].iloc[0]) * 100
    pbi_bm_norm.append(grp)

pbi_bm = pd.concat(pbi_bm_norm)
pbi_bm.to_csv(f"{DASH}/pbi_benchmark.csv", index=False)
print(f" Benchmark         : {len(pbi_bm):,} rows")

 Benchmark         : 8,050 rows


In [9]:
pbi_fol = df_fol.copy()
pbi_fol["month_name"] = pbi_fol["month"].dt.strftime("%b %Y")
pbi_fol["year"]       = pbi_fol["month"].dt.year

pbi_fol.to_csv(f"{DASH}/pbi_folio_count.csv", index=False)
print(f" Folio Count       : {len(pbi_fol)} rows")

 Folio Count       : 21 rows


In [10]:
# Single row KPI table for Power BI cards
kpi = {
    "total_industry_aum_lakh_cr"   : round(total_aum, 2),
    "peak_sip_inflow_crore"        : int(pbi_sip["sip_inflow_crore"].max()),
    "latest_sip_inflow_crore"      : int(pbi_sip["sip_inflow_crore"].iloc[-1]),
    "total_folios_crore"           : float(pbi_fol["total_folios_crore"].iloc[-1]),
    "total_schemes"                : int(df_fund["amfi_code"].nunique()),
    "total_fund_houses"            : int(df_fund["fund_house"].nunique()),
    "total_investors"              : int(df_txn["investor_id"].nunique()),
    "total_transactions"           : int(len(df_txn)),
    "avg_sharpe_ratio"             : round(df_score["sharpe_ratio"].mean(), 3),
    "best_3y_return_pct"           : round(df_master["cagr_3y_pct"].max(), 2),
    "active_sip_accounts_crore"    : float(pbi_sip["active_sip_accounts_crore"].iloc[-1]),
}

pd.DataFrame([kpi]).to_csv(f"{DASH}/pbi_kpi_summary.csv", index=False)

print(" KPI Summary saved")
print("\n KEY METRICS:")
for k, v in kpi.items():
    print(f"   {k:<40} : {v}")

 KPI Summary saved

 KEY METRICS:
   total_industry_aum_lakh_cr               : 62.74
   peak_sip_inflow_crore                    : 31002
   latest_sip_inflow_crore                  : 31002
   total_folios_crore                       : 26.12
   total_schemes                            : 40
   total_fund_houses                        : 10
   total_investors                          : 5000
   total_transactions                       : 32778
   avg_sharpe_ratio                         : 0.537
   best_3y_return_pct                       : 35.11
   active_sip_accounts_crore                : 9.35


In [11]:
exports = [
    "pbi_fund_performance.csv",
    "pbi_monthly_nav.csv",
    "pbi_aum_trends.csv",
    "pbi_sip_trends.csv",
    "pbi_transactions.csv",
    "pbi_category_inflows.csv",
    "pbi_benchmark.csv",
    "pbi_folio_count.csv",
    "pbi_kpi_summary.csv",
]

print("="*55)
print("POWER BI EXPORT FILES")
print("="*55)
for f in exports:
    path = f"bluestock/dashboard/{f}"
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f" {f:<40} {len(df):>8,} rows")
    else:
        print(f" MISSING — {f}")

print(f"\n All files saved to bluestock/dashboard/")

POWER BI EXPORT FILES
 pbi_fund_performance.csv                       40 rows
 pbi_monthly_nav.csv                         2,120 rows
 pbi_aum_trends.csv                             90 rows
 pbi_sip_trends.csv                             48 rows
 pbi_transactions.csv                       32,778 rows
 pbi_category_inflows.csv                      144 rows
 pbi_benchmark.csv                           8,050 rows
 pbi_folio_count.csv                            21 rows
 pbi_kpi_summary.csv                             1 rows

 All files saved to bluestock/dashboard/


In [12]:
import subprocess

cmds = [
    ["git", "-C", "bluestock", "add", "."],
    ["git", "-C", "bluestock", "commit", "-m",
     "Day 5: Power BI exports ready — 9 dashboard CSVs"],
    ["git", "-C", "bluestock", "push", "origin", "main"],
]
for cmd in cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or r.stderr)


[main ffb1f5e] Day 5: Power BI exports ready â€” 9 dashboard CSVs
 9 files changed, 43301 insertions(+)
 create mode 100644 dashboard/pbi_aum_trends.csv
 create mode 100644 dashboard/pbi_benchmark.csv
 create mode 100644 dashboard/pbi_category_inflows.csv
 create mode 100644 dashboard/pbi_folio_count.csv
 create mode 100644 dashboard/pbi_fund_performance.csv
 create mode 100644 dashboard/pbi_kpi_summary.csv
 create mode 100644 dashboard/pbi_monthly_nav.csv
 create mode 100644 dashboard/pbi_sip_trends.csv
 create mode 100644 dashboard/pbi_transactions.csv

To https://github.com/gauravhkulkarni24-ship-it/MutualFund_capstone.git
   6e1e056..ffb1f5e  main -> main

